In [ ]:
# Scenario 1: Operational Baseline Analysis
**IoT Network Forensics Master's Thesis**

The extracted Zeek metadata is then ingested into a Python data analysis environment utilizing the Pandas library. In this phase, the raw datasets are cleansed of environmental background noise, such as standard Mininet overhead and Layer 2 broadcast traffic. The dataset is filtered to isolate the flows specifically associated with the target IoT IP addresses and the gateway. Finally, critical behavioral metrics are mathematically derived from the timestamps, most notably the Inter-Arrival Time (IAT) between specific device communications.

**Topology Definition:**
* Smart Camera: `10.0.0.1`
* Smart Thermostat: `10.0.0.2`
* Attacker Node: `10.0.0.3`
* Cloud/Gateway: `10.0.0.254`
* Duration: 1200 seconds (20 minutes)

: 

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from zat.log_to_dataframe import LogToDataFrame

# Set academic plotting style for LaTeX embedding
sns.set_theme(style="whitegrid")
plt.rcParams.update({
    "font.family": "serif",
    "axes.labelsize": 12,
    "font.size": 12,
    "legend.fontsize": 10,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "figure.dpi": 300
})

# Define paths
LOG_DIR = "../../zeek_logs/scenario1_logs/"  # Adjust relative path if needed
OUTPUT_DIR = "../../figures/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

: 

## 1. Data Ingestion & Preprocessing
We utilize the Zeek Analysis Tools (`zat`) library to safely parse the TSV logs, automatically handling the commented headers and converting Zeek's epoch timestamps into native Python Datetime indices. We then filter out any non-IoT subnet noise (e.g., Mininet IPv6/multicast background traffic).

In [ ]:
# Initialize ZAT parser
log_to_df = LogToDataFrame()

# Ingest Zeek Logs
conn_df = log_to_df.create_dataframe(os.path.join(LOG_DIR, 'conn.log'))
dns_df = log_to_df.create_dataframe(os.path.join(LOG_DIR, 'dns.log'))

# Preprocessing: Ensure timestamps are timezone-naive for easier math
if conn_df.index.tz is not None:
    conn_df.index = conn_df.index.tz_localize(None)
if dns_df.index.tz is not None:
    dns_df.index = dns_df.index.tz_localize(None)

# Define our forensic scope
target_ips = ['10.0.0.1', '10.0.0.2', '10.0.0.3', '10.0.0.254']

# Filter conn.log to only include traffic where our topology nodes are the originators
conn_df = conn_df[conn_df['id.orig_h'].isin(target_ips)]
dns_df = dns_df[dns_df['id.orig_h'].isin(target_ips)]

print(f"Total connections ingested: {len(conn_df)}")
print(f"Total DNS queries ingested: {len(dns_df)}")

## 2. Thermostat Behavior (Keep-Alives & Telemetry)
**Objective:** The Smart Thermostat (`10.0.0.2`) connects to the MQTT broker on port `8883`. 
Over 1200 seconds, we expect exactly 120 Keep-Alive packets (10s intervals) and exactly 4 Telemetry reports (300s intervals). We isolate these flows and calculate the Inter-Arrival Time (IAT).

In [ ]:
# Filter for Thermostat MQTT traffic
thermostat_mqtt = conn_df[(conn_df['id.orig_h'] == '10.0.0.2') & (conn_df['id.resp_p'] == 8883)].copy()

# Sort by timestamp (index) to accurately calculate IAT
thermostat_mqtt = thermostat_mqtt.sort_index()

# Separate Keep-Alives from Telemetry based on payload size (orig_bytes)
# NOTE: Adjust the byte threshold (e.g., 100) based on your specific MUD PCAP profile
keep_alives = thermostat_mqtt[thermostat_mqtt['orig_bytes'] < 100].copy()
telemetry = thermostat_mqtt[thermostat_mqtt['orig_bytes'] >= 100].copy()

# Mathematical Assertions (Ground Truth Validation)
assert len(keep_alives) == 120, f"Expected 120 keep-alives, found {len(keep_alives)}"
assert len(telemetry) == 4, f"Expected 4 telemetry flows, found {len(telemetry)}"
print("✅ Thermostat Flow Counts Validated: 120 Keep-Alives, 4 Telemetry Reports.")

# Calculate Inter-Arrival Time (IAT) in seconds
keep_alives['IAT'] = keep_alives.index.to_series().diff().dt.total_seconds()

iat_mean = keep_alives['IAT'].mean()
iat_var = keep_alives['IAT'].var()

print(f"Keep-Alive IAT Mean: {iat_mean:.4f} seconds")
print(f"Keep-Alive IAT Variance: {iat_var:.4f}")

## 3. Camera Streaming (Volumetric Flow Analysis)
**Objective:** The Smart Camera (`10.0.0.1`) transmits a continuous UDP video stream to port `50005`. We will aggregate the volumetric flow (bytes per minute) to visualize the consistency of the stream.

In [ ]:
# Filter for Camera UDP Video Stream
camera_udp = conn_df[(conn_df['id.orig_h'] == '10.0.0.1') & 
                     (conn_df['id.resp_p'] == 50005) & 
                     (conn_df['proto'] == 'udp')].copy()

total_bytes = camera_udp['orig_bytes'].sum()
print(f"✅ Camera Stream Total Originated Bytes: {total_bytes:,.0f} Bytes")

# Resample volumetric flow per minute
# Note: Zeek conn.log logs the aggregate flow when the connection closes or times out. 
# For a continuous UDP stream, it might be a single large row or segmented based on Zeek's timeout settings.
volume_over_time = camera_udp['orig_bytes'].resample('1T').sum()

# Plotting the Volumetric Flow
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(volume_over_time.index, volume_over_time.values / 1024, marker='o', linestyle='-', color='#1f77b4')
ax.set_title("Smart Camera UDP Video Stream Volumetric Flow")
ax.set_xlabel("Time (MM:SS)")
ax.set_ylabel("Kilobytes (KB)")
ax.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%M:%S'))

plt.tight_layout()
fig_path = os.path.join(OUTPUT_DIR, "camera_volumetric_flow.pdf")
plt.savefig(fig_path, format='pdf', bbox_inches='tight')
plt.show()

## 4. Camera TLS Telemetry & Network Services (NTP & DNS)
**Objective:** Validate the exact frequency of background network services supporting the IoT ecosystem.
* Camera TLS (`443`): Exactly 4 flows.
* NTP (`123`): Exactly 4 flows (2 Camera, 2 Thermostat).
* DNS (`53`): Exactly 40 queries (20 Camera, 20 Thermostat).

In [ ]:
# 4.1 Camera TLS Telemetry
camera_tls = conn_df[(conn_df['id.orig_h'] == '10.0.0.1') & (conn_df['id.resp_p'] == 443)]
assert len(camera_tls) == 4, f"Expected 4 TLS flows, found {len(camera_tls)}"
print("✅ Camera TLS Telemetry Validated: 4 Flows.")

# 4.2 Network Services (NTP)
ntp_flows = conn_df[(conn_df['id.orig_h'].isin(['10.0.0.1', '10.0.0.2'])) & 
                    (conn_df['id.resp_p'] == 123) & 
                    (conn_df['proto'] == 'udp')]
assert len(ntp_flows) == 4, f"Expected 4 NTP flows, found {len(ntp_flows)}"
camera_ntp = len(ntp_flows[ntp_flows['id.orig_h'] == '10.0.0.1'])
therm_ntp = len(ntp_flows[ntp_flows['id.orig_h'] == '10.0.0.2'])
print(f"✅ NTP Synchronization Validated: Total 4 Flows ({camera_ntp} Camera, {therm_ntp} Thermostat).")

# 4.3 DNS Queries
dns_queries = dns_df[dns_df['id.orig_h'].isin(['10.0.0.1', '10.0.0.2'])]
assert len(dns_queries) == 40, f"Expected 40 DNS queries, found {len(dns_queries)}"
camera_dns = len(dns_queries[dns_queries['id.orig_h'] == '10.0.0.1'])
therm_dns = len(dns_queries[dns_queries['id.orig_h'] == '10.0.0.2'])
print(f"✅ DNS Queries Validated: Total 40 Queries ({camera_dns} Camera, {therm_dns} Thermostat).")

## 5. Attacker Dormancy Validation
**Objective:** Scenario 1 represents the "Operational Baseline" prior to the attack phase. Therefore, the attacker node (`10.0.0.3`) must exhibit zero network activity to ensure mathematical purity of the control dataset.

In [ ]:
attacker_ip = '10.0.0.3'

# Check if the attacker sent or received any traffic
attacker_originated = conn_df[conn_df['id.orig_h'] == attacker_ip]
attacker_responded = conn_df[conn_df['id.resp_h'] == attacker_ip]

total_attacker_flows = len(attacker_originated) + len(attacker_responded)

assert total_attacker_flows == 0, f"Baseline Contamination! Found {total_attacker_flows} flows involving the Attacker IP."
print("✅ Attacker Dormancy Validated: 0 packets originating or destined to 10.0.0.3.")